In [65]:
%load_ext autoreload
%autoreload 2

from functions import *
import pandas as pd
import pickle
import os
from datetime import datetime
from evaluate import *

In [33]:
# ticker_list = ['REE', 'SAM', 'HAP', 'GMD', 'GIL', 'TMS', 'SAV', 'DHA', 'MHC', 'HAS'] # 10 stocks with the most observations
ticker_list = ['REE', 'SAM', 'HAP'] # 3 stocks with the most observations
limits = {
    'hose':0.07,
    'hnx':0.1,
    'upcom':0.15
}

# Parameters

horizon = 10
seq_len = 1000
train_pct = 0.958

# Read and merge into 1 dataset

if "stock_data.csv" in os.listdir("data"):
    merged_df = pd.read_csv(
        os.path.join("data", "stock_data.csv"),
        index_col=None
    ).assign(
        date = lambda df : pd.to_datetime(df["date"])
    )
else:
    # Read and merge data
    hnx = pd.read_csv(os.path.join("data", "CafeF.HNX.Upto31.07.2025.csv")).assign(
        floor = "hnx"
    )
    hsx = pd.read_csv(os.path.join("data", "CafeF.HSX.Upto31.07.2025.csv")).assign(
        floor = "hose"
    )
    upcom = pd.read_csv(os.path.join("data", "CafeF.UPCOM.Upto31.07.2025.csv")).assign(
        floor = "upcom"
    )
    indexes = pd.read_csv(os.path.join("data", "CafeF.INDEX.Upto06.08.2025.csv")).assign(
        floor = "index"
    )

    # Rename columns
    hnx, hsx, upcom, indexes = [
        df.rename(columns={
            "<Ticker>":"ticker",
            "<DTYYYYMMDD>":"date",
            "<Open>":"open",
            "<High>":"high",
            "<Low>":"low",
            "<Close>":"close",
            "<Volume>":"volume"
        }) for df in [hnx, hsx, upcom, indexes]
    ]
        
    # Merge and clean data
    # UPCOM has missing tickers for some reason
    merged_df = pd.concat(
        [hnx, hsx, upcom, indexes],
        axis=0
    ).reset_index(drop=True).dropna(subset="ticker")\
    .assign(
        date=lambda df : df["date"].astype(str).apply(lambda x: datetime.strptime(x, "%Y%m%d").date())
    )
    merged_df.to_csv(
        os.path.join("data", "stock_data.csv"),
        index=False
    ) # Save merged data to save time in future runs


# Data cleaning and merging

data = merged_df.sort_values(["ticker", "date"]).assign(
    returns = lambda df : df.groupby("ticker")["close"].pct_change(),
    log_returns_pct = lambda df : np.log(df["close"] / df.groupby("ticker")["close"].shift(1))*100
)

data = data.loc[data["ticker"].str.len()==3] # Eliminate ETF, and indeces

data["limit"] = data["floor"].map(limits)
outliers = data.loc[data["returns"].abs() > data["limit"]]
clean_df = data.drop(outliers.index) # Remove outliers
print(f"% of observations removed: {round((len(outliers)/len(data))*100, 2)}%")

pivoted_data = clean_df.pivot_table(
    columns="ticker", 
    values=["open", "high", "low", "close", "returns"], 
    index="date"
)
pivoted_data.columns = pivoted_data.columns.swaplevel(0, 1)
pivoted_data = pivoted_data.sort_index(axis=1, level=0)
pivoted_data = pivoted_data.loc[:, pivoted_data.columns.get_level_values(0).isin(ticker_list)]
pivoted_data = pivoted_data.dropna() # Drop NA

data_returns = pivoted_data.loc[:, pivoted_data.columns.get_level_values(1) == "returns"]

% of observations removed: 1.05%


In [34]:
train_df, test_df = split_train_test(data_returns, train_ratio = train_pct)
realized_cov = get_rolling_realized_covariance(data_returns.values, window_size=1000)
actual_covs = realized_cov[-len(test_df):]

In [36]:
with open(r"bekk_results\bekk_pred_covs_3A_10S.pkl", "rb") as f:
    bekk_covs = pickle.load(f)
with open(r"dcc_results\dcc_pred_covs_1000SL_3A_10S.pkl", "rb") as f:
    dcc_covs = pickle.load(f)
with open(r"svr_results\svr_pred_covs_3A_10S.pkl", "rb") as f:
    svr_covs = pickle.load(f)

In [ ]:
# Rearrange covariance lists

bekk_eval = np.concatenate(np.array(bekk_covs), axis=0)[:len(actual_covs)]
dcc_eval = np.concatenate(np.array(dcc_covs), axis=0)[:len(actual_covs)]
svr_eval = np.concatenate(np.array(svr_covs), axis=0)[:len(actual_covs)]

In [ ]:
# Calculate Frobenius Loss

bekk_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, bekk_eval)])
dcc_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, dcc_eval)])
svr_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, svr_eval)])

In [ ]:
# Calculate Sharpe Ratio
# There was 

rf = 0.032 # Assuming risk-free rate


In [114]:
scale_factor = 10000 # Add a scale factor to avoid equal weighted covariance

bekk_sum = np.array([np.sum(item, axis=0) for item in np.array(bekk_covs)]) * scale_factor
dcc_sum = np.array([np.sum(item, axis=0) for item in np.array(dcc_covs)]) * scale_factor
svr_sum = np.array([np.sum(item, axis=0) for item in np.array(svr_covs)]) * scale_factor

In [115]:
bekk_ws = [minimum_variance_portfolio(cov_mat, data_returns)[0] for cov_mat in bekk_sum]
dcc_ws = [minimum_variance_portfolio(cov_mat, data_returns)[0] for cov_mat in dcc_sum]
svr_ws = [minimum_variance_portfolio(cov_mat, data_returns)[0] for cov_mat in svr_sum]

In [116]:
bekk_ws

[array([1.00000000e+00, 0.00000000e+00, 2.58628674e-11]),
 array([0.00000000e+00, 1.11022302e-16, 1.00000000e+00]),
 array([0.33442284, 0.35610128, 0.30947588]),
 array([0.        , 0.23749771, 0.76250229]),
 array([3.36699569e-01, 3.46944695e-17, 6.63300431e-01]),
 array([0.49214249, 0.50556461, 0.0022929 ]),
 array([7.00958078e-01, 6.27816727e-05, 2.98979141e-01]),
 array([0.15772371, 0.44064601, 0.40163027]),
 array([4.82518947e-01, 2.38524478e-18, 5.17481053e-01]),
 array([9.36290803e-01, 4.44522891e-18, 6.37091973e-02]),
 array([0.38264597, 0.15839873, 0.4589553 ]),
 array([0.69666694, 0.15166653, 0.15166653]),
 array([9.01630944e-01, 5.55111512e-17, 9.83690562e-02]),
 array([0.25632055, 0.4873589 , 0.25632055]),
 array([0.2664704 , 0.39219957, 0.34133003]),
 array([6.34331476e-12, 0.00000000e+00, 1.00000000e+00]),
 array([0.41289461, 0.10886174, 0.47824365]),
 array([9.71445147e-17, 8.32667268e-16, 1.00000000e+00]),
 array([0.26669495, 0.2353611 , 0.49794394]),
 array([2.49800181

In [ ]:
eval_table = pd.DataFrame({
    "Models":['BEKK-GARCH', "DCC-GARCH", "SVR"],
    "Frobenius Loss": [bekk_frob, dcc_frob, svr_frob]
}).sort_values(
    "Frobenius Loss",
    ascending=True,
    ignore_index=True
)
display(eval_table)

,Models,Frobenius Loss
0,SVR,0.000001
1,DCC-GARCH,0.000002
2,BEKK-GARCH,0.000002
